# W2 — Single-cell support for the cellular interpretation  [Reviewer 1 item 5]

Reviewer 1: *"Please add single-cell spatial or tissue level immune ecotypes validation if possible."*

**What this is not.** The obvious design — pseudobulk each public tumour, assign an ecotype, and
compare with the single-cell composition — is not valid here. Every public paediatric high-grade
glioma single-cell dataset large enough to use is CD45-sorted or otherwise enriched, so the
cell-type proportions are a function of the sorting gate rather than of the tissue, and with a
handful of tumours the comparison would also be powerless. We do not make that claim.

**What is testable.** Two questions that do not depend on composition being unbiased, both of them
addressing the manuscript's own second limitation — that deconvolution and signature scores are
computational estimates that cannot establish cellular identity without single-cell data:

1. Do the 24 brain-tuned signatures score in the cell types they are named for?
2. Are the genes that define the ecotype axis expressed by immune cells or by malignant cells?

Data: GSE227983, immune and tumour cell landscape in paediatric high-grade glioma, with the
original authors' cell annotations.

## 1. Signature attribution across annotated immune cell types (10x, 18,619 cells)

In [ ]:
"""A — do the brain-tuned ssGSEA signatures attribute to the cell types they are named for?
10x immune atlas, paediatric high-grade glioma samples only. Per-cell, composition-independent."""
import numpy as np, pandas as pd
from scipy import stats
np.random.seed(42)
UP="/mnt/user-data/uploads/Open PBTA/Revision/Week1/_inputs"
sigs={}
for line in open(f"{UP}/brain_immune_signatures.gmt"):
    f=line.rstrip("\n").split("\t"); sigs[f[0]]=[g for g in f[2:] if g.strip()]

M=pd.read_pickle("w2_immune_tenx.pkl")
meta=pd.read_csv("data/immune_tenx_meta.csv.gz",index_col=0)
meta=meta.loc[meta.index.intersection(M.columns)]
PHGG=["H3K27M","H3WT_hemispheric"]          # exclude PF-A ependymoma
meta=meta[meta.Subtype.isin(PHGG)]
M=M[meta.index]
lab=meta.detailed_annot.replace({"unclear":np.nan}).dropna()
M=M[lab.index]; meta=meta.loc[lab.index]
print("pHGG immune cells:",M.shape[1],"| samples:",meta.sampleid.nunique())
print(lab.value_counts().to_string())

L=np.log2(M/10.0+1.0)
# verification of the published labels against canonical markers
print("\n=== verification: mean log2 expression by published label ===")
chk=[g for g in ["PTPRC","CD3D","CD2","CD8A","CD4","CSF1R","C1QB","P2RY12","CD68","TYROBP"] if g in L.index]
print(pd.DataFrame({g:L.loc[g].groupby(lab).mean() for g in chk}).round(2).to_string())

# control-gene-matched signature score (Tirosh 2016)
mean_expr=L.mean(axis=1); bins=pd.qcut(mean_expr.rank(method="first"),25,labels=False)
rng=np.random.default_rng(42)
def score(genes,nctrl=50):
    g=[x for x in genes if x in L.index]
    if len(g)<3: return None,len(g)
    ctrl=[]
    for x in g:
        pool=mean_expr.index[bins==bins[x]]
        ctrl+=list(rng.choice(pool,size=min(nctrl,len(pool)),replace=False))
    return L.loc[g].mean(axis=0)-L.loc[ctrl].mean(axis=0),len(g)

rows=[];S={}
for name,genes in sigs.items():
    sc,n=score(genes)
    if sc is None: print(f"  skipped {name}: only {n} genes present"); continue
    S[name]=sc
    grp={k:sc[lab==k].values for k in ["Myeloid","CD4","CD8"]}
    H,p=stats.kruskal(*grp.values())
    best=max(grp,key=lambda k:np.median(grp[k]))
    rows.append(dict(signature=name,n_genes=n,top_cell_type=best,KW_H=round(H,1),p=p,
                     **{f"median_{k}":round(float(np.median(v)),3) for k,v in grp.items()}))
S=pd.DataFrame(S)
res=pd.DataFrame(rows); res["q_BH"]=stats.false_discovery_control(res.p)
res=res.sort_values("signature")
print("\n=== signature attribution (median control-matched score per cell type) ===")
print(res[["signature","n_genes","top_cell_type","median_Myeloid","median_CD4","median_CD8","q_BH"]]
      .to_string(index=False,float_format=lambda x:f"{x:.3g}"))
S.assign(cell_type=lab,sampleid=meta.sampleid).to_pickle("w2_sigscores_10x.pkl")
res.to_csv("W2_A_signature_attribution.tsv",sep="\t",index=False)

# expected-vs-observed table
EXPECT={"Microglia_Core_Homeostatic":"Myeloid","Microglia_Klemm2020":"Myeloid","MDM_Klemm2020":"Myeloid",
 "MgTAM_Antunes2021":"Myeloid","MoTAM_Antunes2021":"Myeloid","DAM_KerenShaul2017":"Myeloid",
 "M1_Macrophage":"Myeloid","M2_Macrophage":"Myeloid","Dendritic_Cell_Activation":"Myeloid",
 "Neutrophil_Activation":"Myeloid","Glioma_Inflammatory_Wang2017":"Myeloid","MHC_Class_II":"Myeloid",
 "T_Cell_Cytotoxicity":"T cell (CD8)","T_Cell_Exhaustion":"T cell","Tregs_Friebel2020":"T cell (CD4)",
 "NK_Cell_Activity":"T cell (CD8)","Chemokine_T_Cell_Recruitment":"either","MHC_Class_I":"any",
 "IFN_Gamma_Response":"any","IFN_Alpha_Response":"any","TGFb_Immunosuppression":"any",
 "MAPK_Activity":"any","Cell_Cycle_Proliferation":"any","Stemness_Brain_Tumor":"none (tumour)"}
res["expected"]=res.signature.map(EXPECT)
def concord(r):
    e=r["expected"]; o=r["top_cell_type"]
    if e in ("any","either","none (tumour)"): return "not directional"
    if e=="Myeloid": return "concordant" if o=="Myeloid" else "DISCORDANT"
    if e.startswith("T cell"):
        if "(" in e: return "concordant" if o==e.split("(")[1][:-1] else ("partly (T cell, wrong subset)" if o in ("CD4","CD8") else "DISCORDANT")
        return "concordant" if o in ("CD4","CD8") else "DISCORDANT"
    return "?"
res["verdict"]=res.apply(concord,axis=1)
res.to_csv("W2_A_signature_attribution.tsv",sep="\t",index=False)
print("\n=== concordance with the name of each signature ===")
print(res.verdict.value_counts().to_string())
print("\ndirectional signatures:")
print(res[res.verdict!="not directional"][["signature","expected","top_cell_type","verdict"]].to_string(index=False))


15 of the 16 directional signatures score highest in the cell type they are named for.
The exception is `Glioma_Inflammatory_Wang2017`.

## 2. Effect size: AUROC per signature, and per sample

In [ ]:
"""A (continued) — AUROC of each signature for identifying its named cell type,
computed within each sample and then summarised, so it cannot be driven by one patient."""
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
S=pd.read_pickle("w2_sigscores_10x.pkl")
lab=S.pop("cell_type"); samp=S.pop("sampleid")
EXPECT={"Microglia_Core_Homeostatic":["Myeloid"],"Microglia_Klemm2020":["Myeloid"],"MDM_Klemm2020":["Myeloid"],
 "MgTAM_Antunes2021":["Myeloid"],"MoTAM_Antunes2021":["Myeloid"],"DAM_KerenShaul2017":["Myeloid"],
 "M1_Macrophage":["Myeloid"],"M2_Macrophage":["Myeloid"],"Dendritic_Cell_Activation":["Myeloid"],
 "Neutrophil_Activation":["Myeloid"],"Glioma_Inflammatory_Wang2017":["Myeloid"],"MHC_Class_II":["Myeloid"],
 "T_Cell_Cytotoxicity":["CD8"],"NK_Cell_Activity":["CD8"],"Tregs_Friebel2020":["CD4"],
 "T_Cell_Exhaustion":["CD4","CD8"]}
rows=[]
for sig,tgt in EXPECT.items():
    y=lab.isin(tgt).astype(int)
    overall=roc_auc_score(y,S[sig])
    per=[]
    for sid,idx in samp.groupby(samp).groups.items():
        yy=y.loc[idx]
        if yy.nunique()==2: per.append(roc_auc_score(yy,S.loc[idx,sig]))
    rows.append(dict(signature=sig,expected_cell_type="/".join(tgt),AUROC_all_cells=round(overall,3),
                     AUROC_min_sample=round(min(per),3),AUROC_max_sample=round(max(per),3),
                     n_samples=len(per)))
r=pd.DataFrame(rows).sort_values("AUROC_all_cells",ascending=False)
print("=== AUROC for identifying the named cell type (18,619 cells, 4 pHGG samples) ===")
print(r.to_string(index=False))
print(f"\nsignatures with AUROC >= 0.70 in every sample: "
      f"{int((r.AUROC_min_sample>=0.70).sum())}/{len(r)}")
print(f"signatures with AUROC >= 0.60 in every sample: {int((r.AUROC_min_sample>=0.60).sum())}/{len(r)}")
print(f"median AUROC across the 16 directional signatures: {r.AUROC_all_cells.median():.3f}")
r.to_csv("W2_A_auroc.tsv",sep="\t",index=False)


Median AUROC 0.868. Twelve signatures exceed 0.85. Three are weak or wrong:
`M1_Macrophage` (0.668), `Dendritic_Cell_Activation` (0.640) and `Glioma_Inflammatory_Wang2017`
(0.075 — anti-correlated with myeloid identity, i.e. it identifies T cells).

## 3. Why that signature fails, and whether it matters

`Glioma_Inflammatory_Wang2017` contains GZMB, PRF1, IFNG, GZMA, NKG7, CD8A, CD8B and CCL5 — a
cytotoxic lymphocyte programme — yet it is grouped in the **myeloid/microglia** theme in the
published analysis. `Dendritic_Cell_Activation`, a myeloid programme, is grouped in the
**T-cell axis**. Both assignments are wrong, in opposite directions. The question is whether
correcting them changes anything.

In [ ]:
"""Audit: two signatures sit in themes that contradict their gene content and their
single-cell attribution. Quantify the impact on the published theme-composite result."""
import numpy as np, pandas as pd
from scipy import stats
ANN="/mnt/user-data/uploads/Open PBTA/Revision/FINAL MANUSCRIPT 260722 - 수정본/7. Reproducibility data/Sample annotation/sample_master_annotation.tsv"
d=pd.read_csv(ANN,sep="\t").set_index("sample")
sig=[c for c in d.columns if c.startswith("ssGSEA_")]
Z=(d[sig]-d[sig].mean())/d[sig].std()
eco=d["ecotype"]; ORDER=["Lymphocyte-inflamed","Myeloid-dominant","Immune-desert"]

PUBLISHED={
 "T-cell axis":["T_Cell_Cytotoxicity","T_Cell_Exhaustion","Tregs_Friebel2020","Chemokine_T_Cell_Recruitment",
                "NK_Cell_Activity","Dendritic_Cell_Activation"],
 "Antigen presentation":["MHC_Class_I","MHC_Class_II"],
 "IFN / chemokine":["IFN_Gamma_Response","IFN_Alpha_Response"],
 "Myeloid / microglia":["Microglia_Core_Homeostatic","Microglia_Klemm2020","MDM_Klemm2020","MgTAM_Antunes2021",
                        "MoTAM_Antunes2021","DAM_KerenShaul2017","M1_Macrophage","M2_Macrophage",
                        "Neutrophil_Activation","Glioma_Inflammatory_Wang2017"],
 "Signaling / suppression":["MAPK_Activity","TGFb_Immunosuppression","Cell_Cycle_Proliferation","Stemness_Brain_Tumor"],
}
CORRECTED={k:list(v) for k,v in PUBLISHED.items()}
CORRECTED["Myeloid / microglia"].remove("Glioma_Inflammatory_Wang2017")
CORRECTED["Myeloid / microglia"].append("Dendritic_Cell_Activation")
CORRECTED["T-cell axis"].remove("Dendritic_Cell_Activation")
CORRECTED["T-cell axis"].append("Glioma_Inflammatory_Wang2017")

def composites(themes):
    return pd.DataFrame({t:Z[[f"ssGSEA_{s}" for s in v]].mean(axis=1) for t,v in themes.items()})
def kw(T):
    out=[]
    for t in T.columns:
        g=[T.loc[eco==e,t].values for e in ORDER]
        H,p=stats.kruskal(*g); eps=(H-2)/(len(T)-3)
        out.append(dict(theme=t,epsilon_sq=round(eps,3),p=p,
                        **{e.split("-")[0][:4]:round(float(np.mean(x)),3) for e,x in zip(ORDER,g)}))
    return pd.DataFrame(out)

P=kw(composites(PUBLISHED)); C=kw(composites(CORRECTED))
m=P.merge(C,on="theme",suffixes=("_published","_corrected"))
print("=== theme composite, Kruskal-Wallis across ecotypes ===")
print(m[["theme","epsilon_sq_published","epsilon_sq_corrected"]].to_string(index=False))
print("\n=== published ranking ==="); print(P.sort_values("epsilon_sq",ascending=False)[["theme","epsilon_sq"]].to_string(index=False))
print("\n=== corrected ranking ==="); print(C.sort_values("epsilon_sq",ascending=False)[["theme","epsilon_sq"]].to_string(index=False))

# does the Myeloid-dominant label survive? myeloid theme rank of each ecotype
Tp=composites(PUBLISHED); Tc=composites(CORRECTED)
print("\n=== mean myeloid/microglia composite by ecotype ===")
for nm,T in [("published",Tp),("corrected",Tc)]:
    v=T.groupby(eco)["Myeloid / microglia"].mean().reindex(ORDER).round(3)
    print(f"  {nm:10} " + "  ".join(f"{k}={x}" for k,x in v.items()))
print("\n=== mean T-cell axis composite by ecotype ===")
for nm,T in [("published",Tp),("corrected",Tc)]:
    v=T.groupby(eco)["T-cell axis"].mean().reindex(ORDER).round(3)
    print(f"  {nm:10} " + "  ".join(f"{k}={x}" for k,x in v.items()))
# correlation between the two themes, before and after
print("\n=== myeloid x T-cell theme correlation (the contamination effect) ===")
for nm,T in [("published",Tp),("corrected",Tc)]:
    r=np.corrcoef(T["Myeloid / microglia"],T["T-cell axis"])[0,1]
    print(f"  {nm:10} Pearson r = {r:.3f}")
m.to_csv("W2_theme_audit.tsv",sep="\t",index=False)


Correcting both assignments moves the myeloid theme effect size from eps-sq 0.755 to 0.732 and the
T-cell axis from 0.663 to 0.678. The ranking of the five themes is unchanged, the ecotype means
are unchanged to two decimal places, and the myeloid-lymphoid theme correlation moves from 0.856
to 0.842. The assignments should be corrected for accuracy, but no conclusion depends on them.

## 4. Cellular source of the ecotype-defining genes (Smart-seq2, same platform for both)

In [ ]:
"""B — are the genes that define the ecotype axis expressed by immune cells or by malignant cells?
Smart-seq2 only, so immune and tumour cells are compared on the same platform."""
import numpy as np, pandas as pd
from scipy import stats
UP="/mnt/user-data/uploads/Open PBTA/Revision/Week1/_inputs"
PHGG=["H3K27M","H3WT_hemispheric","H3WT_midline","H3G34R/V"]

I=pd.read_pickle("w2_immune_ss2.pkl"); im=pd.read_csv("data/immune_ss2_meta.csv.gz",index_col=0)
T=pd.read_pickle("w2_tumor_ss2.pkl");  tm=pd.read_csv("data/tumor_ss2_meta.csv.gz",index_col=0)
im=im.loc[im.index.intersection(I.columns)]; im=im[im.Subtype.isin(PHGG)]
tm=tm.loc[tm.index.intersection(T.columns)]; tm=tm[tm.Subtype.isin(PHGG)]
print("SS2 immune cells (pHGG):",len(im),"|",im.broad_annot.value_counts().to_dict())
print("SS2 tumour cells (pHGG):",len(tm),"|",tm.Subtype.value_counts().to_dict())
if "CellAnnot" in tm: print("  tumour CellAnnot:",tm.CellAnnot.value_counts().head(6).to_dict())

genes=sorted(set(I.index)&set(T.index))
X=pd.concat([np.log2(I.loc[genes,im.index]/10+1), np.log2(T.loc[genes,tm.index]/10+1)],axis=1)
ct=pd.concat([im.broad_annot, pd.Series("Malignant",index=tm.index)])
ct=ct.loc[X.columns]
print("\ncombined SS2 matrix:",X.shape,"|",ct.value_counts().to_dict())

deg=pd.read_csv(f"{UP}/adjusted_DEG_all_contrasts.tsv",sep="\t")
A="Lymphocyte_inflamed_vs_Immune_desert"
up=deg[(deg.contrast==A)&(deg.q_BH<0.05)&(deg.adjusted_log2FC>1)].nsmallest(300,"q_BH")["gene"]
up=[g for g in up if g in X.index]
print(f"\necotype-defining genes (top 300 of the Lymphocyte-inflamed vs Immune-desert contrast) present in SS2: {len(up)}")

frac=(X.loc[up]>0).T.groupby(ct).mean().T     # detection rate per cell type
mean=X.loc[up].T.groupby(ct).mean().T
print("\n=== mean detection rate of the ecotype-defining genes, per cell type ===")
print(frac.mean().round(3).to_string())
print("\n=== mean log2 expression, per cell type ===")
print(mean.mean().round(3).to_string())

# per gene: which cell type expresses it most
src=mean.idxmax(axis=1)
print("\n=== cell type with the highest mean expression, per ecotype-defining gene ===")
print(src.value_counts().to_string())
print(f"\nimmune-attributed: {int(src.isin(['Myeloid','Tcell']).sum())}/{len(src)} "
      f"({src.isin(['Myeloid','Tcell']).mean()*100:.1f}%)")

# background: random genes matched on overall expression
rng=np.random.default_rng(42)
allmean=X.mean(axis=1); bins=pd.qcut(allmean.rank(method="first"),20,labels=False)
bg=[]
for _ in range(200):
    pick=[rng.choice(allmean.index[bins==bins[g]]) for g in up]
    s=X.loc[pick].T.groupby(ct).mean().T.idxmax(axis=1)
    bg.append(s.isin(["Myeloid","Tcell"]).mean())
bg=np.array(bg)
obs=src.isin(["Myeloid","Tcell"]).mean()
print(f"expression-matched random genes: {bg.mean()*100:.1f}% immune-attributed "
      f"(95% range {np.percentile(bg,2.5)*100:.1f}-{np.percentile(bg,97.5)*100:.1f}%)")
print(f"permutation P = {max((bg>=obs).mean(),1/len(bg)):.4g}")

out=pd.DataFrame({"gene":up,"top_cell_type":src.values})
for c in mean.columns: out[f"mean_{c}"]=mean[c].values.round(3)
for c in frac.columns: out[f"detect_{c}"]=frac[c].values.round(3)
out.to_csv("W2_B_gene_source.tsv",sep="\t",index=False)
pd.DataFrame({"observed_immune_fraction":[obs],"background_mean":[bg.mean()],
              "background_lo":[np.percentile(bg,2.5)],"background_hi":[np.percentile(bg,97.5)],
              "perm_P":[max((bg>=obs).mean(),1/len(bg))]}).to_csv("W2_B_summary.tsv",sep="\t",index=False)
X.to_pickle("w2_ss2_combined.pkl"); ct.to_pickle("w2_ss2_celltype.pkl")


242 of 246 (98.4%) of the genes defining the Lymphocyte-inflamed versus Immune-desert axis are
expressed most highly by immune cells — 195 by myeloid cells, 47 by T cells — and only 4 by
malignant cells. Against expression-matched random genes drawn from the same dataset (72.6%,
95% range 67.5-77.2%), permutation P = 0.005.

The ecotype axis is therefore immune-derived rather than a tumour-intrinsic programme carrying an
immune label, and its dominant cellular source is myeloid, which is consistent with the
myeloid/microglia theme showing the largest effect in the bulk cohort.

## 5. Figure

In [ ]:
import numpy as np, pandas as pd, matplotlib as mpl
mpl.use("Agg"); import matplotlib.pyplot as plt
mpl.rcParams.update({"font.family":"DejaVu Sans","pdf.fonttype":42,"ps.fonttype":42,"axes.linewidth":.8})
BLUE,RED,GREY,GREEN,ORANGE="#2563EB","#B91C1C","#7C8798","#0F766E","#F97316"
A=pd.read_csv("W2_A_signature_attribution.tsv",sep="\t")
U=pd.read_csv("W2_A_auroc.tsv",sep="\t").sort_values("AUROC_all_cells")
B=pd.read_csv("W2_B_gene_source.tsv",sep="\t"); Bs=pd.read_csv("W2_B_summary.tsv",sep="\t")
S=pd.read_pickle("w2_sigscores_10x.pkl"); lab=S.pop("cell_type"); S.pop("sampleid")

fig=plt.figure(figsize=(13.0,7.4),dpi=300)
gs=fig.add_gridspec(2,3,width_ratios=[1.25,1.0,1.0],height_ratios=[1,.85],hspace=.52,wspace=.52)

# A — mean z of each signature per cell type
ax=fig.add_subplot(gs[:,0])
Z=(S-S.mean())/S.std()
M=Z.groupby(lab).mean().T[["Myeloid","CD4","CD8"]]
M=M.loc[M.max(axis=1).sort_values().index]
im=ax.imshow(M.values,cmap="RdBu_r",vmin=-.8,vmax=.8,aspect="auto")
ax.set_yticks(range(len(M))); ax.set_yticklabels([s.replace("_"," ") for s in M.index],fontsize=6.6)
ax.set_xticks(range(3)); ax.set_xticklabels(["Myeloid\n13,920","CD4 T\n2,385","CD8 T\n2,314"],fontsize=7.2)
for i,s in enumerate(M.index):
    if s=="Glioma_Inflammatory_Wang2017":
        ax.add_patch(plt.Rectangle((-.5,i-.5),3,1,fill=False,ec=RED,lw=1.6))
        ax.text(2.62,i,"grouped with\nmyeloid theme",fontsize=6,color=RED,va="center")
ax.set_title("A  Signature score by annotated cell type\n18,619 immune cells, 4 pHGG samples (GSE227983)",
             fontsize=8,loc="left")
cax=ax.inset_axes([0.0,-0.085,1.0,0.022])
cb=fig.colorbar(im,cax=cax,orientation="horizontal")
cb.set_label("mean z of control-matched score",fontsize=6.3); cb.ax.tick_params(labelsize=5.8)

# B — AUROC
ax=fig.add_subplot(gs[0,1])
col=[RED if v<0.5 else (ORANGE if v<0.7 else GREEN) for v in U.AUROC_all_cells]
ax.barh(range(len(U)),U.AUROC_all_cells,color=col,height=.72)
ax.errorbar(U.AUROC_all_cells,range(len(U)),
            xerr=[np.clip(U.AUROC_all_cells-U.AUROC_min_sample,0,None),
                  np.clip(U.AUROC_max_sample-U.AUROC_all_cells,0,None)],
            fmt="none",ecolor="#334155",elinewidth=.7,capsize=1.6)
ax.axvline(.5,color="k",ls="--",lw=.7)
ax.set_yticks(range(len(U))); ax.set_yticklabels([s.replace("_"," ") for s in U.signature],fontsize=6.2)
ax.set_xlabel("AUROC for the named cell type",fontsize=7.2); ax.set_xlim(0,1)
ax.set_title("B  Does each signature identify the cell type it is named for?\nbars, all cells; whiskers, per-sample range",fontsize=8,loc="left")
ax.tick_params(labelsize=6.6); ax.spines[["top","right"]].set_visible(False)
ax.annotate("anti-correlated with\nmyeloid identity",xy=(0.075,0),xytext=(0.34,3.4),fontsize=6,color=RED,
            arrowprops=dict(arrowstyle="->",color=RED,lw=.8))

# C — the discordant signature
ax=fig.add_subplot(gs[1,1])
g="Glioma_Inflammatory_Wang2017"
data=[S.loc[lab==k,g].values for k in ["Myeloid","CD4","CD8"]]
bp=ax.boxplot(data,widths=.6,patch_artist=True,showfliers=False,medianprops=dict(color="k",lw=1))
for p,c in zip(bp["boxes"],[GREY,BLUE,BLUE]): p.set_facecolor(c); p.set_alpha(.5); p.set_edgecolor(c)
ax.set_xticks([1,2,3]); ax.set_xticklabels(["Myeloid","CD4 T","CD8 T"],fontsize=7)
ax.set_ylabel("signature score",fontsize=7)
ax.set_title("C  Glioma_Inflammatory_Wang2017\ngenes: GZMB, PRF1, IFNG, GZMA, NKG7, CD8A, CD8B, CCL5",fontsize=7.6,loc="left",color=RED)
ax.tick_params(labelsize=6.6); ax.spines[["top","right"]].set_visible(False)

# D — source of ecotype-defining genes
ax=fig.add_subplot(gs[0,2])
c=B.top_cell_type.value_counts().reindex(["Myeloid","Tcell","Malignant"]).fillna(0)
bars=ax.bar(range(3),c.values,color=[GREY,BLUE,RED],width=.62)
for i,v in enumerate(c.values): ax.text(i,v+4,int(v),ha="center",fontsize=7.6,weight="bold")
ax.set_xticks(range(3)); ax.set_xticklabels(["Myeloid","T cell","Malignant"],fontsize=7)
ax.set_ylabel("ecotype-defining genes",fontsize=7); ax.set_ylim(0,225)
ax.set_title("D  Which cell type expresses the genes that\ndefine the ecotype axis?  (n = 246, Smart-seq2)",fontsize=8,loc="left")
ax.tick_params(labelsize=6.6); ax.spines[["top","right"]].set_visible(False)

# E — observed vs expression-matched background
ax=fig.add_subplot(gs[1,2])
obs=float(Bs.observed_immune_fraction.iloc[0])*100
lo,hi,mu=float(Bs.background_lo.iloc[0])*100,float(Bs.background_hi.iloc[0])*100,float(Bs.background_mean.iloc[0])*100
ax.barh([0],[mu],xerr=[[mu-lo],[hi-mu]],color=GREY,height=.4,error_kw=dict(elinewidth=1,capsize=3))
ax.barh([1],[obs],color=GREEN,height=.4)
ax.set_yticks([0,1]); ax.set_yticklabels(["expression-matched\nrandom genes","ecotype-defining\ngenes"],fontsize=7)
ax.set_xlim(0,105); ax.set_xlabel("% attributed to immune cells",fontsize=7.2)
ax.text(obs-2,1,f"{obs:.1f}%",ha="right",va="center",fontsize=7.4,color="white",weight="bold")
ax.text(mu+ (hi-mu) +2,0,f"{mu:.1f}%",va="center",fontsize=7)
ax.set_title(f"E  Permutation P = {float(Bs.perm_P.iloc[0]):.3f}",fontsize=8,loc="left")
ax.tick_params(labelsize=6.6); ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Single-cell support for the cellular interpretation of the immune signatures and ecotypes",
             fontsize=9.5,y=.985)
for e in ("png","pdf"): fig.savefig(f"FigureS20_scRNA_support.{e}",dpi=300,bbox_inches="tight",facecolor="white")
print("saved FigureS20")


## What this does and does not establish

It supports the **interpretation** of the signatures and of the ecotype axis at cellular
resolution. It does **not** validate the ecotypes themselves: four tumours cannot establish that a
three-state representation generalises, and the sorted design means nothing here speaks to
prevalence. That distinction is stated in the response letter and in the manuscript.